In [ ]:
import os
import warnings
from pathlib import Path

# Set writable cache locations before importing Spateo/Numba-backed modules.
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp/cache-spateo")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-spateo")
os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/numba-spateo")

import anndata as ad
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spateo as st

# Spateo's digitize() currently emits many pandas chained-assignment
# FutureWarnings while filling adata.obs. They are noisy but not fatal here.
warnings.filterwarnings("ignore", category=FutureWarning, module="spateo.digitization.grid")


In [ ]:
data_dir = Path("../data")
xy = pd.read_csv(data_dir / "ca3_xy.csv")
coords = xy[["xcoord", "ycoord"]].to_numpy(dtype=float)

adata = ad.AnnData(X=np.ones((coords.shape[0], 1)))
adata.obs_names = [f"spot_{i + 1}" for i in range(coords.shape[0])]
adata.var_names = ["dummy"]
adata.obsm["spatial"] = coords
adata.obs["xcoord"] = coords[:, 0]
adata.obs["ycoord"] = coords[:, 1]
adata.uns["__type"] = "UMI"

print(adata)
xy.describe()


In [ ]:
xy.head()
#Use these rows

In [ ]:
# The tutorial's digitize() works on image-like ROI coordinates with four user-defined corners.
# Here we use the raw xy tissue coordinates to create a PCA-aligned image coordinate system:
#   axis 0 / columns = longitudinal direction
#   axis 1 / layers  = radial direction
center = coords.mean(axis=0)
coords_centered = coords - center
_, _, vt = np.linalg.svd(coords_centered, full_matrices=False)

longitudinal_axis = vt[0]
radial_axis = vt[1]
longitudinal_score = coords_centered @ longitudinal_axis
radial_score = coords_centered @ radial_axis

# Toggle these after looking at the plots if you want either coordinate direction reversed.
flip_longitudinal = False
flip_radial = False

if flip_longitudinal:
    longitudinal_score = -longitudinal_score
if flip_radial:
    radial_score = -radial_score

def normalize01(x):
    x = np.asarray(x, dtype=float)
    return (x - x.min()) / (x.max() - x.min())

longitudinal01 = normalize01(longitudinal_score)
radial01 = normalize01(radial_score)

# Rescale to a square image-like coordinate system. Keeping both axes within the same
# range avoids axis-shape assumptions inside Spateo's image PDE solver.
#
# Spateo samples the generated field as field[row, col], while OpenCV contours are
# represented as (x=col, y=row). Keep separate arrays so both conventions are correct.
digitize_pad = 50
digitize_scale = 120
longitudinal_img = digitize_pad + digitize_scale * longitudinal01
radial_img = digitize_pad + digitize_scale * radial01

spatial_digitize = np.column_stack([radial_img, longitudinal_img])
contour_digitize = np.column_stack([longitudinal_img, radial_img])

adata.obsm["spatial_digitize"] = spatial_digitize
adata.obs["pca_longitudinal_score"] = longitudinal_score
adata.obs["pca_radial_score"] = radial_score

print("longitudinal_axis:", longitudinal_axis)
print("radial_axis:", radial_axis)
print("spatial_digitize row/col min/max:", spatial_digitize.min(axis=0), spatial_digitize.max(axis=0))
print("contour_digitize x/y min/max:", contour_digitize.min(axis=0), contour_digitize.max(axis=0))


In [ ]:
# Build the region-of-interest contour from the transformed tissue point cloud.
# Use a filled convex hull so every tissue spot is inside the ROI. If the final
# plots look too broad, replace this with a hand-drawn or alpha-shape contour.
img_size = int(np.ceil(max(spatial_digitize.max(), contour_digitize.max()))) + digitize_pad + 1
roi_img = np.zeros((img_size, img_size), dtype=np.uint8)

hull_points = np.round(contour_digitize).astype(np.int32).reshape(-1, 1, 2)
hull = cv2.convexHull(hull_points)
cv2.drawContours(roi_img, [hull], contourIdx=0, color=255, thickness=cv2.FILLED)

roi_img_closed = roi_img

contours, hierarchy = cv2.findContours(
    roi_img_closed,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_NONE,
)

if len(contours) == 0:
    raise RuntimeError("No ROI contour found. Increase point_radius or close_kernel size.")

areas = np.array([cv2.contourArea(c) for c in contours])
ctr_idx = int(np.argmax(areas))

# st.dd.digitize sizes its PDE field from max(adata.obsm[spatial_key]), while
# the drawn contour can extend beyond the observed points by point_radius.
# Keep contour vertices inside the field Spateo will allocate.
field_rowmax = int(np.floor(spatial_digitize[:, 0].max()))
field_colmax = int(np.floor(spatial_digitize[:, 1].max()))
contours = tuple(
    np.stack(
        [
            np.clip(c[:, :, 0], 0, field_colmax),
            np.clip(c[:, :, 1], 0, field_rowmax),
        ],
        axis=2,
    ).astype(np.int32)
    for c in contours
)

print(f"Found {len(contours)} contours; using ctr_idx={ctr_idx}, area={areas[ctr_idx]:.1f}")


In [ ]:
# Match the tutorial's four-corner convention:
#   pnt_xy: min layer, min column
#   pnt_xY: max layer, min column
#   pnt_Xy: min layer, max column
#   pnt_XY: max layer, max column
# In this notebook: column = longitudinal; layer = radial.
# Important: Spateo's field_contours() requires each pnt_* value to be an
# exact coordinate on contours[ctr_idx], not just a nearby tissue spot.
contour = contours[ctr_idx][:, 0, :]
contour_min = contour.min(axis=0)
contour_max = contour.max(axis=0)
contour01 = (contour - contour_min) / (contour_max - contour_min)

def nearest_corner(column_target, layer_target):
    d2_contour = (contour01[:, 0] - column_target) ** 2 + (contour01[:, 1] - layer_target) ** 2
    contour_idx = int(np.argmin(d2_contour))
    point = tuple(map(int, contour[contour_idx]))

    d2_spot = (contour_digitize[:, 0] - point[0]) ** 2 + (contour_digitize[:, 1] - point[1]) ** 2
    spot_idx = int(np.argmin(d2_spot))
    return point, spot_idx

pnt_xy, idx_xy = nearest_corner(0, 0)
pnt_xY, idx_xY = nearest_corner(0, 1)
pnt_Xy, idx_Xy = nearest_corner(1, 0)
pnt_XY, idx_XY = nearest_corner(1, 1)

corner_points = {
    "pnt_xy": pnt_xy,
    "pnt_xY": pnt_xY,
    "pnt_Xy": pnt_Xy,
    "pnt_XY": pnt_XY,
}
corner_indices = {
    "pnt_xy": idx_xy,
    "pnt_xY": idx_xY,
    "pnt_Xy": idx_Xy,
    "pnt_XY": idx_XY,
}

print("Spateo corner points in contour coordinates (x=longitudinal, y=radial):")
for name, point in corner_points.items():
    print(f"  {name}: {point}, spot={adata.obs_names[corner_indices[name]]}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

axes[0].scatter(contour_digitize[:, 0], contour_digitize[:, 1], s=8, alpha=0.5)
axes[0].plot(contour[:, 0], contour[:, 1], color="black", linewidth=1)
for name, point in corner_points.items():
    axes[0].scatter(point[0], point[1], s=80)
    axes[0].text(point[0], point[1], name, fontsize=9)
axes[0].set_title("PCA-aligned ROI contour and Spateo corners")
axes[0].set_aspect("equal")

axes[1].scatter(coords[:, 0], coords[:, 1], s=8, alpha=0.5)
for name, idx in corner_indices.items():
    axes[1].scatter(coords[idx, 0], coords[idx, 1], s=80)
    axes[1].text(coords[idx, 0], coords[idx, 1], name, fontsize=9)
axes[1].set_title("Same corner spots in original xy coordinates")
axes[1].set_aspect("equal")

plt.show()


In [ ]:
# This follows the tutorial's "Digitize the region of interest" section.
# Spateo writes radial/layer heat to digital_layer and longitudinal/column heat to digital_column.
st.dd.digitize(
    adata=adata,
    ctrs=contours,
    ctr_idx=ctr_idx,
    pnt_xy=pnt_xy,
    pnt_xY=pnt_xY,
    pnt_Xy=pnt_Xy,
    pnt_XY=pnt_XY,
    spatial_key="spatial_digitize",
    dgl_layer_key="spateo_radial_raw",
    dgl_column_key="spateo_longitudinal_raw",
    max_itr=5000,
    lh=1,
    hh=100,
)

adata.obs["spateo_radial"] = normalize01(adata.obs["spateo_radial_raw"].astype(float).to_numpy())
adata.obs["spateo_longitudinal"] = normalize01(
    adata.obs["spateo_longitudinal_raw"].astype(float).to_numpy()
)

digitized_coords = pd.DataFrame(
    {
        "xcoord": coords[:, 0],
        "ycoord": coords[:, 1],
        "spateo_longitudinal": adata.obs["spateo_longitudinal"].to_numpy(),
        "spateo_radial": adata.obs["spateo_radial"].to_numpy(),
        "spateo_longitudinal_raw": adata.obs["spateo_longitudinal_raw"].to_numpy(),
        "spateo_radial_raw": adata.obs["spateo_radial_raw"].to_numpy(),
        "pca_longitudinal_score": longitudinal_score,
        "pca_radial_score": radial_score,
    },
    index=adata.obs_names,
)

for key in ["spateo_longitudinal_raw", "spateo_radial_raw", "spateo_longitudinal", "spateo_radial"]:
    values = adata.obs[key].astype(float).to_numpy()
    print(f"{key}: zeros={(values == 0).sum()}, min={values.min():.4f}, max={values.max():.4f}")

digitized_coords.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

sc0 = axes[0].scatter(
    coords[:, 0],
    coords[:, 1],
    c=adata.obs["spateo_longitudinal"],
    s=10,
    cmap="viridis",
)
axes[0].set_title("Spateo longitudinal coordinate")
axes[0].set_aspect("equal")
plt.colorbar(sc0, ax=axes[0], fraction=0.046, pad=0.04)

sc1 = axes[1].scatter(
    coords[:, 0],
    coords[:, 1],
    c=adata.obs["spateo_radial"],
    s=10,
    cmap="magma",
)
axes[1].set_title("Spateo radial coordinate")
axes[1].set_aspect("equal")
plt.colorbar(sc1, ax=axes[1], fraction=0.046, pad=0.04)

plt.show()

digitized_coords.describe()


In [ ]:
#Save longitudinal and radial coordinates for later use

digitized_coords.head()

#Save digitized coords
digitized_coords.to_csv(data_dir / "ca3_xy_digitized.csv", index=True)